In [1]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -U "langchain-core"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [3]:
!pip install -qU langchain-huggingface

Build a semantic search engine with LangChain
https://docs.langchain.com/oss/python/langchain/knowledge-base

In [4]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.5 MB/s eta 0:00:00


In [5]:
# access Gcloud Storage
import datetime
project_id = 'DataPractionerEvaluation'
bucket_name = 'pauldeadmanbertpdf'
from google.colab import auth
auth.authenticate_user()

In [6]:
!gcloud storage cp gs://pauldeadmanbertpdf/symantec-data-loss-prevention-help-center-25-1.pdf .

Copying gs://pauldeadmanbertpdf/symantec-data-loss-prevention-help-center-25-1.pdf to file://./symantec-data-loss-prevention-help-center-25-1.pdf

Average throughput: 30.6MiB/s


To take a quick anonymous survey, run:
  $ gcloud survey



Example: How to generate sample documents when desired:

In [7]:
from langchain_core.documents import Document

example_documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

Let’s load a PDF into a sequence of Document objects.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "symantec-data-loss-prevention-help-center-25-1.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

/tmp/ipykernel_1063/1501524544.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


2192


In [9]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Symantec Data Loss Prevention Help Center 25.1
 
 
Version 25.1
 
 
 Last updated: January 28, 2026

{'producer': 'Apache FOP Version 2.1', 'creator': 'Apache FOP Version 2.1', 'creationdate': '2026-01-28T15:38:39+00:00', 'source': 'symantec-data-loss-prevention-help-center-25-1.pdf', 'total_pages': 2192, 'page': 0, 'page_label': '1'}


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

7328


sentence-transformers/all-mpnet-base-v2

In [11]:
#!pip install -qU langchain-huggingface

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[-0.04017502814531326, 0.05984224006533623, 0.022988026961684227, 0.04219130426645279, -0.012630405835807323, -0.008252167142927647, 0.04492940008640289, -0.037874944508075714, -0.01383455190807581, 0.028869742527604103]


In [14]:
import datetime

x = datetime.datetime.now()
print(x)

2026-07-24 07:43:35.464361


In [15]:
#!pip install -U "langchain-core"

In [16]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [17]:
ids = vector_store.add_documents(documents=all_splits)

Return documents based on similarity to a string query:

In [18]:
results = vector_store.similarity_search(
    "How to block https web uploads using agent or network?"
)

print(results[0])

page_content='• Upload a sensitive file or folder with encrypted files with browsers using HTTPS on Windows endpoints
When a user uploads a sensitive file or folder using a browser, the DLP Agent blocks a user action and automatically
encrypts the file with an .html extension and replaces the original file at the source location. A user is then prompted to
upload this encrypted file or folder using the browser to protect sensitive information.
The maximum supported file size for the Endpoint Prevent: Encrypt response action is 150 MB.
About response rule actions
For information about the Endpoint Prevent: Encrypt response rule action, Response rule best practices
 1636' metadata={'producer': 'Apache FOP Version 2.1', 'creator': 'Apache FOP Version 2.1', 'creationdate': '2026-01-28T15:38:39+00:00', 'source': 'symantec-data-loss-prevention-help-center-25-1.pdf', 'total_pages': 2192, 'page': 1635, 'page_label': '1636', 'start_index': 2497}


In [19]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("How to block https web uploads using agent or network?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.6270148360733089

page_content='• Upload a sensitive file or folder with encrypted files with browsers using HTTPS on Windows endpoints
When a user uploads a sensitive file or folder using a browser, the DLP Agent blocks a user action and automatically
encrypts the file with an .html extension and replaces the original file at the source location. A user is then prompted to
upload this encrypted file or folder using the browser to protect sensitive information.
The maximum supported file size for the Endpoint Prevent: Encrypt response action is 150 MB.
About response rule actions
For information about the Endpoint Prevent: Encrypt response rule action, Response rule best practices
 1636' metadata={'producer': 'Apache FOP Version 2.1', 'creator': 'Apache FOP Version 2.1', 'creationdate': '2026-01-28T15:38:39+00:00', 'source': 'symantec-data-loss-prevention-help-center-25-1.pdf', 'total_pages': 2192, 'page': 1635, 'page_label': '1636', 'start_index': 2497}


Return documents based on similarity to an embedded query:

In [20]:
embedding = embeddings.embed_query("How to block https web uploads using agent or network?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='• Upload a sensitive file or folder with encrypted files with browsers using HTTPS on Windows endpoints
When a user uploads a sensitive file or folder using a browser, the DLP Agent blocks a user action and automatically
encrypts the file with an .html extension and replaces the original file at the source location. A user is then prompted to
upload this encrypted file or folder using the browser to protect sensitive information.
The maximum supported file size for the Endpoint Prevent: Encrypt response action is 150 MB.
About response rule actions
For information about the Endpoint Prevent: Encrypt response rule action, Response rule best practices
 1636' metadata={'producer': 'Apache FOP Version 2.1', 'creator': 'Apache FOP Version 2.1', 'creationdate': '2026-01-28T15:38:39+00:00', 'source': 'symantec-data-loss-prevention-help-center-25-1.pdf', 'total_pages': 2192, 'page': 1635, 'page_label': '1636', 'start_index': 2497}


In [21]:
def search(txt):
  results = vector_store.similarity_search_with_score(txt)
  print("Length of results:",len(results))
  for result in results:
    doc, score = result
    print("===================")
    print(f"Score: {score}\n")
    print(doc.page_content)
    print("PDF Array Element=",doc.metadata.get("page"))


In [22]:
search("Block file web uploads using agent?")

Length of results: 4
Score: 0.660857081413269

across multiple HTTP messages. If the files are uploaded in chunks, Symantec Web Prevent detects the
offending content but does not block the offending content from upload.
For DropBox, files that are over 8 MB are uploaded in chunks.  For OneDrive and Google Drive, files that are
over 1 MB are uploaded in chunks.
You may see different results with different browsers.
You can configure a message for your users to inform them why the sensitive data was blocked. The message appears in
the message parameter of the detection response.
To configure the Data-in-Motion (DIM) REST API action
1. Configure a response rule at the Configure Response Rule screen.
Configuring response rules
2. Add the Block Data-in-Motion action type from the Actions list.
The system displays the Block Data-in-Motion field.
Configuring response rule actions
3. Configure the Block Data-in-Motion parameter.
Block Data-in-Motion configuration parameter
4. Click Save to sav

In [23]:
search("What are the new features?")

Length of results: 4
Score: 0.6015573740005493

This content provides enough detail to help you understand the features. Feature descriptions provide links to detailed
deployment information, where applicable. Specific implementation or configuration details for these new features are not
provided.
 66
PDF Array Element= 65
Score: 0.40135258436203003

• Enforce server ID
• Enforce server version
• OS version
• Total number of CPUs
• Total amount of RAM (GB)
Group Rule Metrics toggle button Adds the following metrics to the local telemetry report:
• Number of active policies by group rule type
Incident Metrics toggle button Adds the following metrics to the local telemetry report:
• Is Data Access Governance being used?
• Is Data Insight being used?
• Total number of custom attributes
• Total number of incidents
• Total number of incidents in database
• Total number of incidents in external storage
• Total number of lookup plugins
Policy Group Metrics toggle button Adds the following me

In [24]:
print(len(docs))
print(docs[66])

2192
page_content='NOTE
The versioning of DLP is changing with Symantec Data Loss Prevention 25.1. The first part (25 in this release)
refers to the year, and the second part (.1 in this release) refers to the versions released in the year. For more
information on the new versioning scheme for DLP, see Release Changes.
• Enforce Server Features in DLP 25.1
• Platform Features in DLP 25.1
• Endpoint Features in DLP 25.1
• Linux Agent Updates in DLP 25.1
• Discover Features in DLP 25.1
• Detection Features in DLP 25.1
• Removed and Deprecated Features in 25.1
• Changes to Discover Scan Support
• Release Changes
Enforce Server Features in DLP 25.1
Learn about new and changed Enforce Server features in Symantec Data Loss Prevention 25.1.
New and Updated Symantec Data Loss Prevention APIs
Recycle Function for All Detection Servers
Modernized System Events and Agent Events Screens
Integrate the Enforce Server with CyberArk for Network Discover Credential Management
Improved Syslog Support
Im

In [25]:
results = vector_store.similarity_search_with_score("What are the new features?")
print("Length of results:",len(results))
doc, score = results[0]
print("===================")
print(f"Score: {score}\n")
print(type(doc))
print("------------------")
print(doc.page_content)
print("------------------")
print(doc.metadata)
print(type(doc.metadata))
print("Page=",doc.metadata.get("page"))

Length of results: 4
Score: 0.6015573740005493

<class 'langchain_core.documents.base.Document'>
------------------
This content provides enough detail to help you understand the features. Feature descriptions provide links to detailed
deployment information, where applicable. Specific implementation or configuration details for these new features are not
provided.
 66
------------------
{'producer': 'Apache FOP Version 2.1', 'creator': 'Apache FOP Version 2.1', 'creationdate': '2026-01-28T15:38:39+00:00', 'source': 'symantec-data-loss-prevention-help-center-25-1.pdf', 'total_pages': 2192, 'page': 65, 'page_label': '66', 'start_index': 1574}
<class 'dict'>
Page= 65
